## Testing rotational distance in either $\color{forestgreen}{\theta}$ or $\color{orchid}{\varphi}$

---

1) generate function $f$ and rotated version $g$ (in either $\theta$ or $\varphi$)
2) compute rotational distance by shifting column (slices) of the projection tensor

---

Things to check in this notebook are if the volume element is included correctly. (ignoring scaling constants for now)

---

In [ ]:
### standard libraries
import time
import numpy as np
import matplotlib.pyplot as plt

### my modules
from fast_volume_align_ot import utils, plots, transforms, distances, signals

In [ ]:
def translate_3d_new(array, phid, p, thid):

    tmp = np.roll(array, (phid, 0, 0), axis=(0))
    res = np.zeros(array.shape, dtype=np.float64) #they are real, right?
    
    for k in range(res.shape[0]):
        for j in range(res.shape[2]):
            if j-thid >= 0:
                jj = j - thid
                kk = k
            else:
                jj = thid-j
                kk = (k-phid + res.shape[0]//2) % res.shape[0]
                
            res[k,:,j] = array[kk, :, jj]
            
    return res

In [ ]:
N = 68
sigma = 2

f = utils.generate_centered_gaussian(N=N, d=3, sigma=sigma)
f_1 = transforms.translate_3d(f, 0, 10, 0)
f_2 = transforms.translate_3d(f, 0, -15, -20)
f_3 = transforms.translate_3d(f, 10, 0, 10)
F = f_1 + f_2 + f_3

# F = transforms.translate_3d(f, 0, 10, 0) + transforms.translate_3d(f, 0, -10, 0)

# F = transforms.translate_3d(f, 0, 20, 0)

# F = f

F = signals.Signal(F, d=3).preprocess_signals()[0]

thetas = np.linspace(0, np.pi, N, endpoint=True)
phis = np.linspace(0, 2*np.pi, N, endpoint=False)

# omega = np.deg2rad(0)
# theta = np.deg2rad(137)
# phi = 10 * 2*np.pi / N

omega = np.deg2rad(0)
theta = np.deg2rad(0)
phi = np.deg2rad(267)

G = transforms.rotate_voxel_zyz(F, omega, theta, phi, order=1)
G = signals.Signal(G, d=3).preprocess_signals()[0]

print('\t original')
plots.orthogonal_2D_projection_plot(F)
print('\t rotated')
plots.orthogonal_2D_projection_plot(G)

In [ ]:
U = transforms.Transform(F, d=3, apply_ramp=False).inverse_cdf_transform()[0]
V = transforms.Transform(G, d=3, apply_ramp=False).inverse_cdf_transform()[0]

In [ ]:
n_phi, n_points, n_theta = U.shape

dists_sw_theta = np.zeros(n_theta)
dists_sw_phi = np.zeros(n_phi)

for theta_idx in range(n_theta):
    # V_theta = transforms.translate_3d(V, 0, 0, theta_idx)
    V_theta = translate_3d_new(V, 10, 0, theta_idx)
    # dists_sw_theta[theta_idx] = np.mean((U - V_theta)**2)
    dists_sw_theta[theta_idx] = np.mean( np.mean( (U - V_theta)**2 , axis=1 ) * np.sin(thetas) )
    # dists_sw_theta[theta_idx] = np.mean( np.mean( (U[10, :, :] - V_theta[10, :, :])**2 , axis=0 ) * np.sin(thetas) )

    # U_theta = translate_3d_new(U, 0, 0, theta_idx)
    # dists_sw_theta[theta_idx] = np.mean( np.mean( (U_theta - V)**2 , axis=1 ) * np.sin(thetas) )
    
for phi_idx in range(n_phi):
    V_phi = transforms.translate_3d(V, phi_idx, 0, 0)
    # dists_sw_phi[phi_idx] = np.mean((U - V_phi)**2)
    dists_sw_phi[phi_idx] = np.mean( np.mean( (U - V_phi)**2 , axis=1 ) * np.sin(thetas) )

dists_sw_theta = dists_sw_theta / np.amax(dists_sw_theta)
dists_sw_phi = dists_sw_phi / np.amax(dists_sw_phi)

min_theta = thetas[np.argmin(dists_sw_theta)]
min_phi = phis[np.argmin(dists_sw_phi)]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3))

axs[0].plot(np.rad2deg(thetas), dists_sw_theta, 'p--', color='forestgreen', label=r'$R_\theta$')
axs[0].grid(which='major', linestyle='--')
axs[0].axvline(np.rad2deg(min_theta), linestyle='--', color='gray')
axs[0].set_ylim(-0.1, 1.1)
axs[0].legend()

axs[1].plot(np.rad2deg(phis), dists_sw_phi, 'p--', color='orchid', label=r'$R_\varphi$')
axs[1].grid(which='major', linestyle='--')
axs[1].axvline(np.rad2deg(min_phi), linestyle='--', color='gray')
axs[1].set_ylim(-0.1, 1.1)
axs[1].legend()

plt.tight_layout()
plt.show()

print('\t\t\t min_theta: ', np.round(np.rad2deg(min_theta), 3), '\t\t\t\t\t min_phi: ', np.round(np.rad2deg(min_phi), 3))
print('\t\t\t  gt_theta: ', np.round(np.rad2deg(theta), 3), '\t\t\t\t\t  gt_phi: ', np.round(np.rad2deg(phi), 3))

In [ ]:
np.round(np.rad2deg(thetas), 2)

In [ ]:
np.round(np.rad2deg(phis), 2)